# Exploration initiale - detection de tumeur cerebrale

Ce notebook garde une trace pedagogique de la premiere exploration du projet: chargement d un petit dataset binaire, visualisation, baseline CNN simple et evaluation.

Les pipelines de reference du projet final sont les scripts:
- `train_transfer_learning.py` pour la detection binaire finale
- `train_tumor_type_classifier.py` pour la classification multiclasses
- `app.py` pour l interface Gradio avec Grad-CAM

> Prototype pedagogique uniquement: ce projet ne fournit pas de diagnostic medical.


## 1. Configuration

On fixe les chemins du projet et une graine aleatoire pour rendre les experimentations plus stables.


In [ ]:
from pathlib import Path
import os
import random

SEED = 42
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
MODELS_DIR = BASE_DIR / "models"

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODELS_DIR:", MODELS_DIR)


## 2. Imports

Les imports restent limites aux outils necessaires pour cette exploration baseline.


In [ ]:
import json

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

print("Imports OK")


## 3. Chargement du dataset exploratoire

Ce notebook utilise un petit dataset binaire public pour tester rapidement une premiere approche. Les scripts finaux du projet utilisent des sources plus larges et une strategie de deduplication plus stricte.


In [ ]:
dataset_path = Path(kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection"))
yes_folder = dataset_path / "yes"
no_folder = dataset_path / "no"

yes_images = sorted(list(yes_folder.glob("*.jpg")) + list(yes_folder.glob("*.png")))
no_images = sorted(list(no_folder.glob("*.jpg")) + list(no_folder.glob("*.png")))

print("Dataset:", dataset_path)
print("Images avec tumeur:", len(yes_images))
print("Images sans tumeur:", len(no_images))


## 4. Visualisation rapide

Quelques images sont affichees pour verifier le contenu et la variabilite visuelle du dataset.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
samples = [
    (yes_images[0], "Avec tumeur"),
    (yes_images[1], "Avec tumeur"),
    (no_images[0], "Sans tumeur"),
    (no_images[1], "Sans tumeur"),
]

for ax, (image_path, title) in zip(axes.ravel(), samples):
    image = Image.open(image_path)
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 5. Pretraitement

Les images sont converties en niveaux de gris, redimensionnees, puis normalisees entre 0 et 1.


In [ ]:
IMG_SIZE = 256


def load_grayscale_images(image_paths, label):
    images = []
    labels = []

    for image_path in image_paths:
        try:
            image = Image.open(image_path).convert("L")
            image = image.resize((IMG_SIZE, IMG_SIZE))
            images.append(np.array(image, dtype=np.float32) / 255.0)
            labels.append(label)
        except Exception as error:
            print(f"Image ignoree {image_path}: {error}")

    return images, labels

positive_images, positive_labels = load_grayscale_images(yes_images, label=1)
negative_images, negative_labels = load_grayscale_images(no_images, label=0)

X = np.array(positive_images + negative_images, dtype=np.float32)
y = np.array(positive_labels + negative_labels, dtype=np.int32)

print("X:", X.shape)
print("y:", y.shape)
print("Distribution: tumor=", int(np.sum(y == 1)), "no_tumor=", int(np.sum(y == 0)))


## 6. Split train / validation / test

Le split est stratifie pour conserver la proportion des deux classes dans chaque sous-ensemble.


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp,
)

print("Train:", X_train.shape[0])
print("Validation:", X_val.shape[0])
print("Test:", X_test.shape[0])


## 7. Baseline CNN simple

Ce modele est volontairement simple. Il sert de point de depart exploratoire, pas de modele final du projet.


In [ ]:
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

baseline_model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
])

baseline_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

baseline_model.summary()


## 8. Entrainement baseline

Pour le projet final, utiliser plutot `python3 train_transfer_learning.py`.


In [ ]:
history = baseline_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=16,
    verbose=1,
)


## 9. Evaluation baseline

Les metriques ci-dessous permettent de comparer ce premier essai avec les resultats du pipeline final.


In [ ]:
y_proba = baseline_model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
}

print(json.dumps(metrics, indent=2))
print(classification_report(y_test, y_pred, target_names=["No Tumor", "Tumor"], zero_division=0))


## 10. Sauvegarde optionnelle

Cette sauvegarde concerne uniquement le baseline du notebook. Les modeles finaux sont sauvegardes par les scripts d entrainement.


In [ ]:
baseline_model_path = MODELS_DIR / "brain_tumor_baseline_cnn.keras"
baseline_metrics_path = OUTPUT_DIR / "baseline_cnn_results.json"
baseline_history_path = OUTPUT_DIR / "baseline_cnn_training_history.png"

baseline_model.save(baseline_model_path)
with open(baseline_metrics_path, "w") as file:
    json.dump(metrics, file, indent=4)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train_accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy")

plt.tight_layout()
plt.savefig(baseline_history_path)
plt.show()

print("Modele baseline:", baseline_model_path)
print("Metriques baseline:", baseline_metrics_path)
print("Courbes baseline:", baseline_history_path)
